# 🧠 BraTS 2D JEPA Benchmark — Google Colab GPU Runner
### Complete Training, Fine-Tuning, Probing & Evaluation on Google Colab (Tesla T4 / A100 / V100)

This notebook allows you to train and evaluate the **BraTS 2D JEPA** self-supervised representation learning framework directly in Google Colab.

**Key Highlights:**
- ⚡ **CUDA Mixed Precision (AMP):** Enabled by default (`--amp`) for ~3x faster throughput on Tensor Cores.
- 💾 **Google Drive Persistence:** Automatically connects to your Google Drive so checkpoints, metrics, and figures are permanently saved even if Colab disconnects!
- 🚀 **Fast Local NVMe I/O:** The dataset is unzipped into `/content/` for high-throughput PyTorch DataLoader batching.

## 1. Google Drive Mounting & GPU Check
Mount Google Drive to persist checkpoints and access uploaded datasets, then verify your assigned GPU.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!nvidia-smi

import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
else:
    print("⚠️ No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

## 2. Install Dependencies
Install `monai` for medical image segmentation metrics (HD95, Dice).

In [ ]:
!pip install -q --no-cache-dir monai

## 3. Clone Codebase from GitHub & Editable Install
Clones your GitHub repository `https://github.com/hanriman/2d_mri_brats_2024_segmentation.git` directly into `/content/thesis_2d`.

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/hanriman/2d_mri_brats_2024_segmentation.git"
work_dir = Path("/content/thesis_2d")

if not (work_dir / "src").exists():
    print(f"Cloning latest codebase from public GitHub repo: {REPO_URL} ...")
    !git clone {REPO_URL} {work_dir}
else:
    print(f"Codebase already present at {work_dir}. Pulling latest updates...")
    !git -C {work_dir} pull

# Switch working directory in IPython permanently
%cd /content/thesis_2d

# Install codebase in editable mode
!pip install -q -e .

sys.path.insert(0, str(work_dir / "src"))
print(f"\n✓ Working directory set to: {Path.cwd()}")
print("✓ Active commit (confirming latest fixes):")
!git log -1 --oneline

## 4. Dataset Setup (High-Speed Local Unpacking)
Unpack `brats_2d_datasets.zip` directly onto the local Colab NVMe SSD (`/content/`).

> **Why unzip to `/content/` instead of reading directly from Drive?**
> Google Drive uses a FUSE network mount. Reading thousands of small `.npy` files across network FUSE causes massive I/O latency. Unzipping to `/content/` provides **~10x faster** DataLoader batching!

In [ ]:
import zipfile
from pathlib import Path

# Search for brats_2d_datasets.zip
data_candidates = list(Path("/content/drive/MyDrive").glob("**/brats_2d_datasets.zip")) + list(Path("/content").glob("brats_2d_datasets.zip"))
if data_candidates:
    data_zip = data_candidates[0]
    print(f"Extracting dataset from: {data_zip} to /content/ ...")
    with zipfile.ZipFile(data_zip, 'r') as zf:
        zf.extractall("/content/")
    print("✓ Dataset extraction complete!")
else:
    print("⚠️ brats_2d_datasets.zip not found!")
    print("Please upload brats_2d_datasets.zip to your Google Drive or /content/ directory.")

# Verify paths
from brats_jepa.config import get_metadata_path

gli_meta = get_metadata_path("brats_gli_2d")
print(f"BraTS-GLI-2D Metadata: {gli_meta} (Exists: {gli_meta.exists()})")

## 5. Output Directory & Google Drive Sync Setup
Configure output directory so models, logs, and metrics are written directly to Google Drive.

In [ ]:
drive_output_dir = Path("/content/drive/MyDrive/thesis_2d_outputs")
drive_output_dir.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved permanently to: {drive_output_dir}")

--- 
## 6. Phase 1: Self-Supervised JEPA Pre-training
Pre-train Vision Transformers on 2D slices without labels using Mixed Precision (`--amp`).

In [ ]:
# Pre-train Standard I-JEPA (50 Epochs, AMP)
!python scripts/train_jepa.py --model_type ijepa --epochs 50 --batch_size 32 --num_workers 2 --amp --output_dir /content/drive/MyDrive/thesis_2d_outputs

In [ ]:
# Pre-train SigReg JEPA (50 Epochs, AMP)
!python scripts/train_jepa.py --model_type sigreg_jepa --epochs 50 --batch_size 32 --num_workers 2 --amp --output_dir /content/drive/MyDrive/thesis_2d_outputs

In [ ]:
# Pre-train VisReg JEPA (50 Epochs, AMP)
!python scripts/train_jepa.py --model_type visreg_jepa --epochs 50 --batch_size 32 --num_workers 2 --amp --output_dir /content/drive/MyDrive/thesis_2d_outputs

--- 
## 7. Phase 2: Supervised Baselines (UNet & nnU-Net)

In [ ]:
# Train BraTS 2D UNet Baseline
!python scripts/train_unet.py --epochs 30 --batch_size 32 --num_workers 2 --amp --output_dir /content/drive/MyDrive/thesis_2d_outputs

# Train BraTS 2D nnU-Net Baseline
!python scripts/train_nnunet.py --epochs 30 --batch_size 32 --num_workers 2 --amp --output_dir /content/drive/MyDrive/thesis_2d_outputs

---
## 8. Phase 3: Downstream Segmentation Fine-Tuning
Fine-tune pre-trained JEPA encoders with multiscale decoder pyramid and target encoder weights.

In [ ]:
sep = "=" * 70
for model in ["ijepa", "sigreg_jepa", "visreg_jepa"]:
    print(f"\n{sep}\nFine-Tuning Downstream Segmentation: {model.upper()} (Multiscale Pyramid)\n{sep}")
    !python scripts/train_downstream.py --model_type {model} --epochs 30 --batch_size 32 --num_workers 2 --amp \
        --decoder_type multiscale --encoder_source target \
        --output_dir /content/drive/MyDrive/thesis_2d_outputs \
        --checkpoint_dir /content/drive/MyDrive/thesis_2d_outputs/checkpoints


--- 
## 9. Phase 4: Downstream Evaluation & Probing

In [ ]:
!python scripts/evaluate.py --batch_size 32 --num_workers 2 --amp --decoder_type multiscale \
    --output_dir /content/drive/MyDrive/thesis_2d_outputs \
    --checkpoint_dir /content/drive/MyDrive/thesis_2d_outputs/checkpoints

--- 
## 10. Phase 5: Low-Data Label Efficiency & OOD Benchmarks

In [ ]:
# Low-data benchmark (1% - 100% annotations with 3D volumetric metrics)
!python scripts/evaluate_low_data.py --epochs 30 --batch_size 32 --num_workers 2 --amp \
    --decoder_type multiscale --encoder_source target \
    --output_dir /content/drive/MyDrive/thesis_2d_outputs \
    --checkpoint_dir /content/drive/MyDrive/thesis_2d_outputs/checkpoints \
    --exp_version colab_low_data

# OOD scanner shifts
!python scripts/evaluate_ood.py --batch_size 32 --num_workers 2 --amp \
    --output_dir /content/drive/MyDrive/thesis_2d_outputs \
    --checkpoint_dir /content/drive/MyDrive/thesis_2d_outputs/checkpoints \
    --exp_version colab_ood

# OOD Meningioma zero-shot transfer
!python scripts/evaluate_men_rt_ood.py --max_samples 5000 --batch_size 32 --num_workers 2 --amp \
    --output_dir /content/drive/MyDrive/thesis_2d_outputs \
    --checkpoint_dir /content/drive/MyDrive/thesis_2d_outputs/checkpoints \
    --exp_version colab_men_rt_ood


--- 
## 11. Phase 6: Publication Figures & Inline Display

In [ ]:
!python scripts/generate_figures.py \
    --metrics_dir /content/drive/MyDrive/thesis_2d_outputs/metrics \
    --figures_dir /content/drive/MyDrive/thesis_2d_outputs/figures

from pathlib import Path

from IPython.display import Image, display

fig_dir = Path("/content/drive/MyDrive/thesis_2d_outputs/figures")
for p in sorted(fig_dir.glob("*.png")):
    print(f"\nPlot: {p.name}")
    display(Image(filename=str(p), width=750))